In [3]:
"""
Markov Switching Price Spread Model 42
=======================================
Real-time direct h-step forecasts of real TTF NG prices.
Two regimes: cointegration (alpha < -0.005, beta ≤ -0.001) and
no-cointegration (alpha ≥ -0.005, -0.001 < beta < 0.001).
MCMC: 1000 burn-in, 2000 retained draws (thin=2). Point forecast = median.
max_rejections=100 (vs Baumeister's 5000) — near-zero Regime 1 acceptance rate.
"""

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

SQRT2PI = np.sqrt(2 * np.pi)

HORIZONS        = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START      = "2015-01-01"
TRAIN_START     = "2006-02-01"
PI_BAR_WINDOW   = 120
N_BURN          = 1000
N_KEEP          = 2000
THIN            = 2
MAX_REJECT      = 100
N_REGIMES       = 2
PRICES_FILE     = "Input_Monthly_Average_Oil_NG_nominal_prices.xlsx"
INFLATION_FILE  = "Input_Inflation.xlsx"
REAL_FILE       = "Input_Real_Average_Monthly_TTF_NG_prices.xlsx"
OUTPUT_FILE     = "Output_PriceSpread_MS_forecasts.xlsx"
RANDOM_SEED     = 42
FALLBACK        = {0: (-0.05, -0.05), 1: (-0.004, 0.0)}

# Load data
prices = pd.read_excel(PRICES_FILE, sheet_name="Sheet1")
prices.columns = ["date", "oil_nom", "ng_nom"]
prices["date"] = pd.to_datetime(prices["date"])
prices = prices.sort_values("date").reset_index(drop=True)
prices["s_ng"]   = np.log(prices["ng_nom"])
prices["s_oil"]  = np.log(prices["oil_nom"])
prices["spread"] = prices["s_ng"] - prices["s_oil"]

inf = pd.read_excel(INFLATION_FILE, sheet_name="Sheet1")
inf.columns = ["date", "hicp", "inf_pct_change"]
inf["date"] = pd.to_datetime(inf["date"])
inf = inf.sort_values("date").reset_index(drop=True)
inf["pi_bar"] = inf["inf_pct_change"].rolling(PI_BAR_WINDOW).mean()

prices["prev_month"]   = (prices["date"].dt.to_period("M").dt.to_timestamp()
                          - pd.offsets.MonthBegin(1))
prices["pi_bar"]       = prices["prev_month"].map(inf.set_index("date")["pi_bar"])
prices["hicp_prev"]    = prices["prev_month"].map(inf.set_index("date")["hicp"])
prices["hicp_nowcast"] = prices["hicp_prev"] * (1 + prices["pi_bar"])
prices["real_ng"]      = prices["ng_nom"] / (prices["hicp_nowcast"] / 100)

real_df = pd.read_excel(REAL_FILE, parse_dates=["date"])
real_df["date"] = real_df["date"] + pd.offsets.MonthEnd(0)

def get_actual(ym_str):
    m = real_df[real_df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["price_real"].values[0] if len(m) == 1 else np.nan

prices_train = prices[prices["date"] >= TRAIN_START].dropna(
    subset=["spread", "pi_bar", "real_ng"]).reset_index(drop=True)

def hamilton_filter_smoother(y, sp, alpha, beta, sigma, P):
    n   = len(y)
    mu  = alpha[None, :] + beta[None, :] * sp[:, None]
    sig = np.maximum(sigma[None, :], 1e-8)
    lik = np.exp(-0.5 * ((y[:, None] - mu) / sig) ** 2) / (sig * SQRT2PI)
    lik = np.maximum(lik, 1e-300)
    xi  = np.empty((n, 2)); pred = np.ones(2) / 2
    for t in range(n):
        joint = lik[t] * pred; total = joint.sum()
        xi[t] = joint / max(total, 1e-300); pred = P.T @ xi[t]
    xs = np.empty((n, 2)); xs[-1] = xi[-1]
    for t in range(n - 2, -1, -1):
        pn = np.maximum(P.T @ xi[t], 1e-300)
        xs[t] = xi[t] * (P @ (xs[t + 1] / pn))
        xs[t] /= max(xs[t].sum(), 1e-300)
    return xs

def sample_params(y_k, x_k, regime):
    n = len(y_k)
    if n < 3:
        a, b = FALLBACK[regime]; return a, b, 0.1
    X = np.column_stack([np.ones(n), x_k])
    coef, _, _, _ = np.linalg.lstsq(X, y_k, rcond=None)
    rss = max(((y_k - X @ coef) ** 2).sum(), 1e-10)
    sigma2 = 1.0 / np.random.gamma(n / 2.0, 2.0 / rss)
    sigma  = np.sqrt(max(sigma2, 1e-8))
    try: cov = sigma2 * np.linalg.inv(X.T @ X + np.eye(2) * 1e-6)
    except: cov = np.eye(2) * sigma2 * 0.01
    draws = np.random.multivariate_normal(coef, cov, MAX_REJECT)
    for a, b in draws:
        if regime == 0 and a < -0.005 and b <= -0.001: return a, b, sigma
        if regime == 1 and a >= -0.005 and -0.001 < b < 0.001: return a, b, sigma
    a, b = FALLBACK[regime]; return a, b, sigma

def gibbs_sampler(y_arr, sp_arr, seed_offset=0):
    np.random.seed(RANDOM_SEED + seed_offset)
    n = len(y_arr)
    alpha = np.array([-0.05, -0.002])
    beta  = np.array([-0.05,  0.000])
    sigma = np.array([ 0.15,  0.150])
    P     = np.array([[0.90,  0.10],
                      [0.10,  0.90]])
    keeps = []
    for it in range(N_BURN + N_KEEP * THIN):
        xs = hamilton_filter_smoother(y_arr, sp_arr, alpha, beta, sigma, P)
        u  = np.random.rand(n)
        s  = (u > xs[:, 0]).astype(int)
        for k in range(N_REGIMES):
            idx = (s == k)
            if idx.sum() >= 3:
                alpha[k], beta[k], sigma[k] = sample_params(y_arr[idx], sp_arr[idx], k)
        for k in range(N_REGIMES):
            counts = np.array([((s[:-1]==k)&(s[1:]==j)).sum() for j in range(N_REGIMES)], dtype=float)
            P[k] = np.random.dirichlet(1.0 + counts)
        if it >= N_BURN and (it - N_BURN) % THIN == 0:
            keeps.append((alpha.copy(), beta.copy(), sigma.copy(), P.copy()))
    return keeps

def ms_forecast(keeps, R_t, spread_t, E_pi):
    preds = []
    for alpha_d, beta_d, sigma_d, P_d in keeps:
        try:
            ev, evec = np.linalg.eig(P_d.T)
            pi_stat  = np.abs(evec[:, np.argmin(np.abs(ev - 1.0))])
            pi_stat /= pi_stat.sum()
        except: pi_stat = np.ones(N_REGIMES) / N_REGIMES
        s_h = np.random.choice(N_REGIMES, p=pi_stat)
        fc  = R_t * np.exp(alpha_d[s_h] + beta_d[s_h] * spread_t - E_pi)
        if np.isfinite(fc) and fc > 0: preds.append(fc)
    return float(np.median(preds)) if preds else np.nan

# Main loop
records      = []
eval_origins = prices_train[
    (prices_train["date"] >= EVAL_START) &
    (prices_train["date"] <= "2025-12-31")
]["date"].tolist()

n_total = len(eval_origins) * len(HORIZONS)
t_start = time.time()

print(f"Markov Switching Price Spread (Model 42)")
print(f"Origins: {len(eval_origins)}  |  Horizons: {len(HORIZONS)}  |  Total fits: {n_total}")
print(f"Gibbs: {N_BURN} burn-in + {N_KEEP} draws (thin={THIN}) -> {N_KEEP//THIN} effective")
print(f"Restrictions: Regime 0: alpha<-0.005 & beta<=-0.001  |  Regime 1: alpha>=-0.005 & -0.001<beta<0.001")
print(f"Estimated runtime: ~2-3 hours")
print()

for i, origin_date in enumerate(eval_origins):
    t_orig  = time.time()
    t_idx   = prices_train.index[prices_train["date"] == origin_date][0]
    train   = prices_train.iloc[:t_idx + 1]
    n_train = len(train)
    R_t     = train["real_ng"].iloc[-1]
    spread_t= train["spread"].iloc[-1]
    pi_bar_t= train["pi_bar"].iloc[-1]

    for h in HORIZONS:
        y_vals, sp_vals = [], []
        for j in range(n_train - h):
            y_j  = train["s_ng"].iloc[j + h] - train["s_ng"].iloc[j]
            sp_j = train["spread"].iloc[j]
            if not np.isnan(y_j) and not np.isnan(sp_j):
                y_vals.append(y_j); sp_vals.append(sp_j)
        if len(y_vals) < 10: continue

        y_arr  = np.array(y_vals); sp_arr = np.array(sp_vals)
        E_pi   = h * pi_bar_t
        seed_offset = i * len(HORIZONS) + HORIZONS.index(h)
        try:
            keeps = gibbs_sampler(y_arr, sp_arr, seed_offset=seed_offset)
            fcst  = ms_forecast(keeps, R_t, spread_t, E_pi)
        except Exception: fcst = np.nan

        actual_ym  = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
        records.append({
            "forecast_origin": origin_date.strftime("%Y-%m-%d"),
            "horizon":         h,
            "model":           "PriceSpread(42) MS alpha_MS beta_MS",
            "actual_month":    actual_ym,
            "forecast":        fcst,
            "actual":          get_actual(actual_ym),
        })

    elapsed   = time.time() - t_start
    remaining = elapsed / max(i + 1, 1) * (len(eval_origins) - i - 1)
    print(f"  Origin {i+1:3d}/{len(eval_origins)}: "
          f"{origin_date.strftime('%Y-%m-%d')}  T={n_train}  "
          f"origin={time.time()-t_orig:.0f}s  "
          f"elapsed={elapsed/60:.1f}m  remaining~{remaining/60:.0f}m")

results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

print()
print("=" * 60)
print("MARKOV SWITCHING COMPLETE")
print("=" * 60)
print(f"  Rows: {len(results)}  |  Origins: {results['forecast_origin'].nunique()}")
print(f"  Time: {(time.time()-t_start)/60:.1f} minutes  |  Output: {OUTPUT_FILE}")
first = results["forecast_origin"].min()
print(f"\nSample — first origin ({first}):")
print(results[results["forecast_origin"]==first][["horizon","actual_month","forecast","actual"]].to_string(index=False))


Markov Switching Price Spread (Model 42)
Origins: 132  |  Horizons: 9  |  Total fits: 1188
Gibbs: 1000 burn-in + 2000 draws (thin=2) -> 1000 effective
Restrictions: Regime 0: alpha<-0.005 & beta<=-0.001  |  Regime 1: alpha>=-0.005 & -0.001<beta<0.001
Estimated runtime: ~2-3 hours

  Origin   1/132: 2015-01-31  T=108  origin=344s  elapsed=5.7m  remaining~751m
  Origin   2/132: 2015-02-28  T=109  origin=399s  elapsed=12.4m  remaining~805m
  Origin   3/132: 2015-03-31  T=110  origin=228s  elapsed=16.2m  remaining~696m
  Origin   4/132: 2015-04-30  T=111  origin=408s  elapsed=23.0m  remaining~736m
  Origin   5/132: 2015-05-31  T=112  origin=422s  elapsed=30.0m  remaining~763m
  Origin   6/132: 2015-06-30  T=113  origin=564s  elapsed=39.4m  remaining~828m
  Origin   7/132: 2015-07-31  T=114  origin=540s  elapsed=48.4m  remaining~865m
  Origin   8/132: 2015-08-31  T=115  origin=70s  elapsed=49.6m  remaining~769m
  Origin   9/132: 2015-09-30  T=116  origin=278s  elapsed=54.2m  remaining~741m
